# 01 – Pricing Basics

`PricingEngine` calculates credit costs from usage metrics. Define
model pricing formulas as math expressions, then call `calculate()`.

No storage needed — pure computation.

## Setup

In [1]:
from ducto.engine import PricingEngine
from ducto.metrics import UsageMetrics, ToolCall

### Static config via `from_dict`

Each model maps to an expression with these variables:

`input_tokens`, `output_tokens`, `cache_read_tokens`, `cache_write_tokens` \
`tool_calls`, `search_queries`, `search_results` \
`web_search_calls`, `code_exec_calls`, `fixed_job`

In [2]:
config = {
    "models": {
        "gpt-4o": "input_tokens * 5 + output_tokens * 15",
        "claude-sonnet-4": "input_tokens * 3 + output_tokens * 15",
        "claude-haiku-3.5": "input_tokens * 1 + output_tokens * 4",
    },
    "tools": {"code_exec": "tool_calls * 50"},
    "search": {"costs": "search_queries * 10 + search_results * 1"},
    "cache": {"discount": "cache_read_tokens * 1 + cache_write_tokens * 5"},
}
engine = PricingEngine.from_dict(config)
schema = engine.pricing_schema()
print(f"Engine ready — {len(schema.models)} models registered")

Engine ready — 3 models registered


### Basic call (tokens only)

In [3]:
cost = engine.calculate(UsageMetrics(
    model="gpt-4o", input_tokens=500, output_tokens=200,
))
print(f"  Model:  {cost.model_credits}  ({500}×5 + {200}×15)")
print(f"  Tools:  {cost.tool_credits}")
print(f"  Total:  {cost.total}")
assert cost.total == 5500

  Model:  5500.0  (500×5 + 200×15)
  Tools:  0.0
  Total:  5500.0


### With tool calls

In [4]:
cost = engine.calculate(UsageMetrics(
    model="claude-sonnet-4", input_tokens=1000, output_tokens=400,
    tool_calls=[ToolCall(name="code_exec")],
))
print(f"  Model: {cost.model_credits}  Tools: {cost.tool_credits}  Total: {cost.total}")
assert cost.total == 9050  # 3000+6000+50

  Model: 9000.0  Tools: 50.0  Total: 9050.0


### With search / RAG

In [5]:
cost = engine.calculate(UsageMetrics(
    model="gpt-4o", input_tokens=200, output_tokens=50,
    search_queries=3, search_results=45,
))
print(f"  Model: {cost.model_credits}  Search: {cost.search_credits}  Total: {cost.total}")
assert cost.total == 1825  # 1750 + 75

  Model: 1750.0  Search: 75.0  Total: 1825.0


### With cache discount

In [6]:
cost = engine.calculate(UsageMetrics(
    model="claude-haiku-3.5", input_tokens=300, output_tokens=100,
    cache_read_tokens=200, cache_write_tokens=50,
))
print(f"  Model: {cost.model_credits}  Cache: {cost.cache_savings}  Total: {cost.total}")
print(f"Breakdown keys: {list(cost.breakdown.keys())}")

  Model: 700.0  Cache: 450.0  Total: 1150.0
Breakdown keys: ['model', 'input_tokens', 'output_tokens', 'tool_count']
